# 04 · Noise — `gwmock-noise`

Instrumental noise as composable **components**: coloured Gaussian noise
from a PSD, narrow spectral lines, glitches — stacked in one config.
Everything in this notebook runs in seconds.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260706-leuven-gw-workshop-gwmock/blob/main/materials/notebooks/04-noise.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in the correlated-noise machinery used for the
# stochastic-background examples.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 1. Coloured noise from an ET sensitivity curve

The simulators generate straight into memory (`.generate`) or to files via
config/CLI. ET PSD presets are built in:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gwmock_noise import ColoredNoiseSimulator, estimate_psd

fs = 1024.0
sim = ColoredNoiseSimulator(psd_file="ET_10_full_cryo_psd")
strain = sim.generate(duration=64.0, sampling_frequency=fs, detectors=["E1"], seed=42)
x = np.asarray(strain["E1"])
print("generated", x.size, "samples")

plt.figure(figsize=(9, 2.5))
plt.plot(np.arange(x.size) / fs, x, lw=0.3)
plt.xlabel("time [s]")
plt.ylabel("strain")
plt.title("ET '10 km full cryo' coloured noise")
plt.show()

**Anchor it**: estimate the PSD of what we generated and overlay the
preset curve we asked for — they should track each other.

In [ ]:
from gwmock_noise.spectral import load_and_interpolate_psd

freqs, psd = estimate_psd(x, sampling_frequency=fs)
ref = load_and_interpolate_psd("ET_10_full_cryo_psd", freqs[1:])

plt.figure(figsize=(8, 4))
plt.loglog(freqs[1:], np.sqrt(psd[1:]), lw=0.7, label="measured ASD (Welch)")
plt.loglog(freqs[1:], np.sqrt(ref), "k--", label="requested ET_10_full_cryo")
plt.xlim(3, fs / 2)
plt.xlabel("frequency [Hz]")
plt.ylabel(r"ASD [$1/\sqrt{\rm Hz}$]")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
from gwmock_noise import compare_psd

print("compare_psd says the spectrum matches:",
      compare_psd(x, "ET_10_full_cryo_psd", sampling_frequency=fs))

## 2. Components compose — lines and glitches

A `NoiseConfig` stacks components in order. Coloured noise + a 60 Hz
mains line + parametric blip glitches:

In [ ]:
from pathlib import Path
from gwmock_noise import (BlipGlitch, DefaultNoiseSimulator,
                          LogNormalAmplitudeDistribution, NoiseConfig,
                          OutputConfig, SpectralLine)

config = NoiseConfig(
    detectors=["E1"],
    duration=32.0,
    sampling_frequency=fs,
    components=[
        {"simulator": "colored", "psd_file": "ET_10_full_cryo_psd"},
        {"simulator": "spectral_lines",
         "lines": [SpectralLine(frequency=60.0, amplitude=2e-24)]},
        {"simulator": "glitches",
         "models": [BlipGlitch(rate=0.3, width=0.01,
                               amplitude_distribution=LogNormalAmplitudeDistribution(mean=1.0, std=0.5))]},
    ],
    output=OutputConfig(directory=Path("noise_out"), prefix="messy"),
    seed=42,
)
result = DefaultNoiseSimulator().run(config)
print("written:", result.output_paths)

In [ ]:
y = np.load(result.output_paths["E1"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(np.arange(y.size) / fs, y, lw=0.3)
axes[0].set_xlabel("time [s]")
axes[0].set_title("time series — spot the blips")
f2, p2 = estimate_psd(y, sampling_frequency=fs)
axes[1].loglog(f2[1:], np.sqrt(p2[1:]), lw=0.7)
axes[1].axvline(60, color="r", ls=":", alpha=0.6)
axes[1].set_xlabel("frequency [Hz]")
axes[1].set_title("spectrum — the 60 Hz line")
plt.tight_layout()
plt.show()

Other components you can drop in the same list: `correlated`
(cross-detector CSDs), `ar` (autoregressive colouring), time-varying PSDs
(`psd_schedule`), Schumann-resonance coupling, gengli-based realistic
blips, and real GWOSC data (`gwmock-noise[gwosc]`).

## 3. Streaming — noise without end

Chunks continue seamlessly from simulator state (for memory-bounded or
online use):

In [ ]:
from gwmock_noise import open_stream, take

sim2 = ColoredNoiseSimulator(psd_file="ET_D_psd", detectors=["E1"], sampling_frequency=fs)
stream = open_stream(sim2, chunk_duration=4.0, sampling_frequency=fs, detectors=["E1"])
chunk = next(stream)                     # one 4-s chunk
more = take(stream, total_duration=12.0, chunk_duration=4.0, sampling_frequency=fs)
print("chunk:", chunk["E1"].shape, "+ stream continuation:", more["E1"].shape)

## 4. Built-in QA — `run_diagnostics`

`run_diagnostics` bundles gaussianity + stationarity checks. Since
v0.6.1 it **whitens the data against its own PSD estimate first**, so the
checks are valid for steeply coloured spectra too (the underlying Levene
and KS tests assume nearly independent samples; raw ET-coloured noise
has correlation times of seconds and would misfire — pass
`whiten=False` only for data that is already broadband):

In [ ]:
from gwmock_noise import WhiteNoiseSimulator, run_diagnostics

w = np.asarray(WhiteNoiseSimulator().generate(duration=64.0, sampling_frequency=fs,
                                              detectors=["E1"], seed=1)["E1"])
for label, arr in [("white", w), ("coloured (ET)", x)]:
    verdict = {k: ("PASS" if v.passed else "FAIL") for k, v in
               run_diagnostics(arr, sampling_frequency=fs).items()}
    print(f"{label:14s} {verdict}")

Both pass — and the checks stay sharp: multiply half the series by 3
(non-stationary) or paste in loud glitches (non-Gaussian) and the
corresponding test fails. Together with the `compare_psd` anchor from
section 1, that is a complete QA loop for simulated noise.

## ✏️ Try it (5 min)

1. Compare `ET_10_full_cryo_psd` vs `ET_20_full_cryo_psd` ASDs on one plot
   (generate both, `estimate_psd` each).
2. Add a second `SpectralLine` at 100 Hz with 10× the amplitude, find it
   in the spectrum.
3. Crank `BlipGlitch(rate=...)` to `2.0` and re-plot the time series.

Next: **05 · End-to-end** — population + signal + noise + provenance.